In [ ]:
# Connect to GPU  ---- # control + enter , # shift + return
!pip install -q openai==0.28 PyPDF2 sentence-transformers

In [ ]:
# Importing relevant libraries

import os
import PyPDF2
import openai
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Create open AI Api key
# Set OpenAI API Key
openai.api_key = ''

In [ ]:
# User will give a directory location / folder location
# Theer is a function which will read all the pdf files stored in that folder
# Within that function we will extract texts from all pages and then return the raw data


# Initialize the SentenceTransformer model for embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Function to read and extract text from PDF
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text += page.extract_text()
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
    return text


# Function to split document into smaller chunks (e.g., by paragraphs)
def split_document_into_chunks(document_text, chunk_size=1000):
    # Split the document into paragraphs
    paragraphs = document_text.split("\n")
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) < chunk_size:
            current_chunk += "\n" + para
        else:
            chunks.append(current_chunk.strip())
            current_chunk = para

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


# Function to load documents from a folder and convert them into embeddings
def load_documents_from_folder(folder_path):
    documents = []
    filenames = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            text = extract_text_from_pdf(file_path)
            if text.strip():  # Avoid adding empty texts
                documents.append(text)
                filenames.append(filename)
    return documents, filenames


# Function to generate embeddings for a list of texts
def generate_embeddings(texts):
    embeddings = model.encode(texts, convert_to_numpy=True)
    return embeddings

In [ ]:
# Function to retrieve the most relevant chunk based on a user query
def retrieve_answer_from_documents(query, documents, embeddings, chunk_size=1000):
    # Split documents into smaller chunks
    all_chunks = []
    for doc in documents:
        chunks = split_document_into_chunks(doc, chunk_size)
        all_chunks.extend(chunks)

    # Generate embeddings for all document chunks
    chunk_embeddings = generate_embeddings(all_chunks)

    # Generate the query embedding
    query_embedding = model.encode([query], convert_to_numpy=True)

    # Compute cosine similarity between query and chunk embeddings
    similarities = cosine_similarity(query_embedding, chunk_embeddings)

    # Get the index of the most relevant chunk
    best_match_idx = np.argmax(similarities)
    best_match_chunk = all_chunks[best_match_idx]

    return best_match_chunk


In [ ]:
# Function to generate a detailed answer using OpenAI's GPT model (via the chat completion endpoint)
def generate_answer(query, context):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": query},
        {"role": "assistant", "content": context},
    ]

    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=messages,
        max_tokens=150,
        temperature=0.7
    )

    return response['choices'][0]['message']['content'].strip()

In [ ]:
# Main function to run the Q&A application
def document_qa_rag(folder_path, query):
    # Load documents from folder
    documents, filenames = load_documents_from_folder(folder_path)

    # Retrieve the most relevant chunk from the documents
    best_match_chunk = retrieve_answer_from_documents(query, documents, embeddings=None)

    # Generate a detailed answer from OpenAI based on the best matching chunk
    answer = generate_answer(query, best_match_chunk)

    return answer

In [ ]:
# Example usage
if __name__ == "__main__":
    folder_path = '/content'  # Specify the folder containing your PDFs
    query = input("Enter your question")

    answer = document_qa_rag(folder_path, query)
    print(f"Answer: {answer}")